# Búsqueda no informada

**Facundo A. Lucianna - Inteligencia Artificial - CEIA - FIUBA**

En la notebook anterior armamos la **búsqueda primero en anchura** (BFS) para resolver la Torre de Hanoi. En esta notebook vamos a implementar el resto de los algoritmos de **búsqueda no informada** que vimos en la teoría:

1. Búsqueda primero en profundidad (DFS)
2. Búsqueda de costo uniforme (UCS)
3. Búsqueda de profundidad limitada
4. Búsqueda de profundidad limitada con profundidad iterativa

Todos siguen el mismo esquema que BFS: sacan un nodo de la frontera, se fijan si es el objetivo y, si no lo es, agregan a la frontera sus nodos sucesores. Lo único que cambia entre uno y otro es **en qué orden se sacan los nodos de la frontera**, es decir, qué tipo de cola usamos.

Como en la notebook anterior, todas las funciones devuelven el último nodo de la solución y un diccionario con métricas, para poder compararlas.

In [1]:
import time

from aima_libs.aima import PriorityQueue
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

Como vamos a crear el problema muchas veces, armamos una función que lo hace para una cantidad de discos dada:

In [2]:
def create_problem(number_disks: int = 5) -> ProblemHanoi:
    """Crea el problema de la Torre de Hanoi con todos los discos en la primera
    varilla, y el objetivo de llevarlos a la tercera."""
    list_disks = list(range(number_disks, 0, -1))
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    return ProblemHanoi(initial=initial_state, goal=goal_state)

---

## Búsqueda primero en profundidad (DFS)

La búsqueda primero en profundidad siempre expande el nodo **más profundo** de la frontera. Para eso usamos una cola **LIFO** (una pila): el último nodo que entra es el primero que sale. Es exactamente el mismo código que BFS, pero en lugar de insertar los nodos al principio de la lista con `insert(0, ...)`, los agregamos al final con `append()`.

Igual que en BFS, guardamos los estados ya explorados para no volver a expandirlos. Sin eso, DFS podría quedarse moviendo un disco de un lado a otro para siempre.

In [3]:
def depth_first_search(number_disks: int = 5) -> tuple:
    """Búsqueda primero en profundidad, con registro de estados explorados."""
    problem = create_problem(number_disks)

    frontier = [NodeHanoi(problem.initial)]  # Cola LIFO con el nodo inicial
    explored = set()  # Conjunto de estados ya visitados
    node_explored = 0

    while len(frontier) != 0:
        node = frontier.pop()  # Sacamos el último nodo que entró
        node_explored += 1
        explored.add(node.state)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)  # Lo agregamos al final de la pila

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": 0,
        "max_depth": None,
        "cost_total": None,
    }
    return None, metrics

In [4]:
solution, metrics = depth_first_search(number_disks=5)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 122
states_visited: 122
nodes_in_frontier: 63
max_depth: 121
cost_total: 121.0


DFS encontró una solución expandiendo muy pocos nodos, pero el camino tiene **121 movimientos**, cuando la solución óptima para 5 discos tiene 31 (2⁵ - 1). DFS se mete en la primera rama que encuentra y la sigue hasta el final, sin importarle si hay un camino más corto por otro lado. Por eso **no es óptimo**.

Veamos los primeros movimientos:

In [5]:
for action in solution.solution()[:10]:
    print(action)

Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 3


---

## Búsqueda de costo uniforme (UCS)

La búsqueda de costo uniforme expande siempre el nodo con **menor costo acumulado** desde el inicio, `g(n)`. Para eso usamos una **cola prioritaria**, como la de AIMA que vimos en la notebook anterior, con `g(n)` como prioridad. En nuestros nodos, `g(n)` está guardado en el atributo `path_cost`.

Hay un detalle importante: con una cola prioritaria, un mismo estado puede entrar varias veces a la frontera, por caminos distintos. Por eso nos fijamos si un estado ya fue explorado **al sacarlo** de la frontera, y no solo al agregarlo. La primera vez que sale un estado, sale por el camino más barato.

In [6]:
def uniform_cost_search(number_disks: int = 5) -> tuple:
    """Búsqueda de costo uniforme: expande primero el nodo de menor g(n)."""
    problem = create_problem(number_disks)

    # Cola prioritaria ordenada por el costo acumulado del nodo
    frontier = PriorityQueue(order="min", f=lambda node: node.path_cost)
    frontier.append(NodeHanoi(problem.initial))
    explored = set()
    node_explored = 0

    while len(frontier) != 0:
        _, node = frontier.pop()  # El nodo con menor costo acumulado

        # Si ya lo exploramos por un camino más barato, lo salteamos
        if node.state in explored:
            continue

        node_explored += 1
        explored.add(node.state)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": 0,
        "max_depth": None,
        "cost_total": None,
    }
    return None, metrics

In [7]:
solution, metrics = uniform_cost_search(number_disks=5)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 216
states_visited: 216
nodes_in_frontier: 34
max_depth: 31
cost_total: 31.0


UCS encontró el camino óptimo de 31 movimientos. Es **óptimo** siempre que los costos de las acciones sean positivos. En la Torre de Hanoi, todos los movimientos cuestan lo mismo, así que expandir por menor costo es lo mismo que expandir por menor profundidad: en este problema, UCS encuentra el mismo camino que BFS. La diferencia aparece en problemas donde las acciones tienen costos distintos.

Lo que sí cambia es la cantidad de nodos expandidos: UCS expandió 216, mientras que BFS, en la notebook anterior, expandía unos 1.350. El BFS que armamos solo revisa si un estado ya fue explorado al agregarlo a la frontera, así que un mismo estado puede entrar varias veces y expandirse más de una vez. UCS, en cambio, descarta los estados repetidos al sacarlos. Es una mejora que también le podríamos hacer a BFS.

---

## Búsqueda de profundidad limitada

DFS tiene un problema: si una rama es muy profunda (o infinita), puede perderse en ella. La búsqueda de profundidad limitada es DFS con un **límite de profundidad**: los nodos que están en el límite no se expanden.

En lugar de un conjunto de estados explorados, guardamos para cada estado **la menor profundidad a la que lo vimos**. Si volvemos a un estado por un camino más corto, vale la pena explorarlo de nuevo, porque desde ahí tenemos más profundidad disponible antes de llegar al límite.

In [8]:
def depth_limited_search(number_disks: int = 5, limit: int = 31) -> tuple:
    """Búsqueda primero en profundidad que no expande nodos a más de `limit` pasos
    del inicio."""
    problem = create_problem(number_disks)

    frontier = [NodeHanoi(problem.initial)]
    best_depth = {problem.initial: 0}  # Menor profundidad a la que vimos cada estado
    node_explored = 0

    while len(frontier) != 0:
        node = frontier.pop()
        node_explored += 1

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(best_depth),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        # Los nodos que están en el límite no se expanden
        if node.depth < limit:
            for next_node in node.expand(problem):
                # Solo si es un estado nuevo o lo alcanzamos por un camino más corto
                if next_node.depth < best_depth.get(next_node.state, limit + 1):
                    best_depth[next_node.state] = next_node.depth
                    frontier.append(next_node)

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(best_depth),
        "nodes_in_frontier": 0,
        "max_depth": limit,
        "cost_total": None,
    }
    return None, metrics

Si el límite es menor que la profundidad de la solución más corta, no hay forma de encontrarla:

In [9]:
solution, metrics = depth_limited_search(number_disks=5, limit=20)
print("¿Encontró solución con límite 20?", metrics["solution_found"])

solution, metrics = depth_limited_search(number_disks=5, limit=31)
print("¿Encontró solución con límite 31?", metrics["solution_found"])
print("Largo del camino:", metrics["cost_total"])

¿Encontró solución con límite 20? False
¿Encontró solución con límite 31? True
Largo del camino: 31.0


El problema es que, en general, **no sabemos de antemano** a qué profundidad está la solución. En la Torre de Hanoi lo sabemos porque es un problema muy estudiado, pero en la mayoría de los problemas no.

---

## Búsqueda de profundidad limitada con profundidad iterativa

La solución es no elegir un límite, sino probarlos todos: ejecutamos la búsqueda de profundidad limitada con límite 0, después con 1, después con 2, y así hasta encontrar la solución. Como cada búsqueda explora todos los caminos hasta su límite, la primera solución que aparece está a la **menor profundidad posible**.

In [10]:
def iterative_deepening_search(number_disks: int = 5, max_limit: int = 200) -> tuple:
    """Ejecuta la búsqueda de profundidad limitada con límites cada vez más grandes,
    hasta encontrar una solución."""
    total_explored = 0

    for limit in range(max_limit + 1):
        solution, metrics = depth_limited_search(number_disks, limit)
        total_explored += metrics["nodes_explored"]

        if metrics["solution_found"]:
            # Contamos los nodos expandidos en todas las iteraciones
            metrics["nodes_explored"] = total_explored
            metrics["final_limit"] = limit
            return solution, metrics

    return None, {"solution_found": False, "nodes_explored": total_explored}

In [11]:
solution, metrics = iterative_deepening_search(number_disks=5)

for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 7605
states_visited: 201
nodes_in_frontier: 11
max_depth: 31
cost_total: 31.0
final_limit: 31


La profundidad iterativa encontró el camino óptimo de 31 movimientos, pero expandió muchos más nodos que UCS: cada iteración vuelve a explorar todo lo que exploraron las anteriores. A cambio, en cada momento solo guarda en la frontera los nodos de una rama, como DFS, así que usa mucha menos memoria que BFS o UCS. Cuando no sabemos a qué profundidad está la solución y la memoria es un problema, es una muy buena opción.

---

## Comparando los algoritmos

Para terminar, comparemos los algoritmos con distintas cantidades de discos. Para tener la referencia, volvemos a definir BFS como en la notebook anterior:

In [12]:
def breadth_first_search(number_disks: int = 5) -> tuple:
    """Búsqueda primero en anchura, como en la notebook anterior."""
    problem = create_problem(number_disks)

    frontier = [NodeHanoi(problem.initial)]
    explored = set()
    node_explored = 0

    while len(frontier) != 0:
        node = frontier.pop()
        node_explored += 1
        explored.add(node.state)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.insert(0, next_node)

    return None, {"solution_found": False, "nodes_explored": node_explored}

In [13]:
algorithms = {
    "BFS": breadth_first_search,
    "DFS": depth_first_search,
    "Costo uniforme": uniform_cost_search,
    "Profundidad iterativa": iterative_deepening_search,
}

print(f"{'Algoritmo':<24}{'Discos':>7}{'Movimientos':>13}{'Nodos':>9}{'Tiempo':>10}")
for number_disks in [4, 5, 6]:
    for name, algorithm in algorithms.items():
        start = time.perf_counter()
        _, metrics = algorithm(number_disks=number_disks)
        elapsed = time.perf_counter() - start

        moves = int(metrics["cost_total"])
        nodes = metrics["nodes_explored"]
        print(f"{name:<24}{number_disks:>7}{moves:>13}{nodes:>9}{elapsed:>9.2f}s")
    print()

Algoritmo                Discos  Movimientos    Nodos    Tiempo
BFS                           4           15      216     0.00s
DFS                           4           40       41     0.00s
Costo uniforme                4           15       67     0.00s
Profundidad iterativa         4           15      791     0.01s

BFS                           5           31     1351     0.03s
DFS                           5          121      122     0.00s
Costo uniforme                5           31      216     0.01s


Profundidad iterativa         5           31     7605     0.14s



BFS                           6           63     7246     0.16s
DFS                           6          364      365     0.01s
Costo uniforme                6           63      717     0.02s


Profundidad iterativa         6           63    99883     1.97s



Resumiendo lo que vimos:

| Algoritmo | ¿Encuentra el camino más corto? | Memoria |
|---|---|---|
| BFS | Sí, si todas las acciones cuestan lo mismo | Mucha: guarda todo un nivel del árbol |
| DFS | No | Poca: guarda una rama |
| Costo uniforme | Sí, si los costos son positivos | Mucha |
| Profundidad limitada | No, y si el límite es corto no encuentra nada | Poca |
| Profundidad iterativa | Sí, si todas las acciones cuestan lo mismo | Poca, a cambio de repetir trabajo |

Ninguno de estos algoritmos sabe nada sobre el problema: no tienen forma de saber si un estado está más cerca o más lejos del objetivo. En la próxima notebook vamos a darles esa información con una **función heurística**.